In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip freeze

aisuite==0.1.14
annotated-doc==0.0.4
annotated-types==0.7.0
anyio==4.12.1
asttokens @ file:///C:/b/abs_9662ywy9fp/croot/asttokens_1743630464377/work
beautifulsoup4==4.14.3
certifi==2026.2.25
cffi==2.0.0
charset-normalizer==3.4.4
click==8.3.1
colorama @ file:///C:/Users/dev-admin/perseverance-python-buildout/croot/colorama_1699472650914/work
comm @ file:///C:/miniconda3/conda-bld/comm_1763119289758/work
cryptography==46.0.5
debugpy @ file:///C:/miniconda3/conda-bld/debugpy_1762421659047/work
decorator @ file:///C:/miniconda3/conda-bld/decorator_1757341251593/work
distro==1.9.0
docstring-parser==0.15
executing @ file:///C:/miniconda3/conda-bld/executing_1757061247746/work
fastapi==0.135.1
greenlet==3.3.2
h11==0.16.0
httpcore==1.0.9
httptools==0.7.1
httpx==0.27.2
idna==3.11
ipykernel @ file:///C:/miniconda3/conda-bld/ipykernel_1762946093253/work
ipython @ file:///C:/miniconda3/conda-bld/ipython_1762789324552/work
ipython_pygments_lexers @ file:///C:/b/abs_b66hj1lo19/croot/ipython_pygments

In [4]:
from datetime import datetime
from urllib import response
from aisuite import Client
from src.research_tools import (
    arxiv_search_tool,
    tavily_search_tool,
    wikipedia_search_tool,
)

client = Client()

In [ ]:
# === Agente de Pesquisa (Research Agent) ===
def research_agent(prompt: str, model: str = "openai:gpt-4.1-mini", return_messages: bool = False):
    """
    Agente especializado em coletar e sintetizar informações usando ferramentas externas 
    (Arxiv, Tavily, Wikipedia) via interface do aisuite.
    """
    print("==================================")
    print("start - 🔍 Research Agent")
    print("==================================")

    # Definição do System Prompt: Aqui você configura a "personalidade" e as regras do agente.
    full_prompt = f"""
        You are an advanced research assistant with expertise in information retrieval and academic research methodology. 
        Your mission is to gather comprehensive, accurate, and relevant information on any topic requested by the user.

        AVAILABLE RESEARCH TOOLS:
        1. tavily_search_tool: General web search (news, blogs, websites).
        2. arxiv_search_tool: Academic database (Computer Science, Physics, Mathematics).
        3. wikipedia_search_tool: Historical context and basic definitions.

        RESEARCH METHODOLOGY:
        Analyze the request,
        plan the search, 
        execute it with the tools, 
        evaluate the sources and synthesize.

        OUTPUT FORMAT:
        Retorne APENAS uma lista de fontes com:
        - Title: [Title]
        - URL/Link: [Link]
        - Abstract: [Summary]

        Today is {datetime.now().strftime("%Y-%m-%d")}.

        USER RESEARCH REQUEST:
        {prompt}
    """.strip()

    # Prepara a mensagem para a API
    messages = [{"role": "user", "content": full_prompt}]
    
    # Lista de ferramentas que o modelo tem permissão para "chamar"
    tools = [arxiv_search_tool, tavily_search_tool, wikipedia_search_tool]

    try:
        # Chamada ao modelo via aisuite
        print("Calling the model...")
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto", # O modelo decide sozinho qual ferramenta usar
            max_turns=5,        # Permite que ele faça até 5 "voltas" (perguntar -> usar ferramenta -> responder)
            temperature=0.0,    # Resposta determinística (focada e sem "criatividade" excessiva)
        )
        content = resp.choices[0].message.content or ""

        # ---- Coleta de informações sobre quais ferramentas foram usadas ----
        calls = []

        # A) Busca chamadas de ferramentas nas respostas intermediárias (loops do agente)
        print("Searching for tool calls in intermediate responses...")
        for ir in getattr(resp, "intermediate_responses", []) or []:
            try:
                tcs = ir.choices[0].message.tool_calls or []
                for tc in tcs:
                    calls.append((tc.function.name, tc.function.arguments))
            except Exception:
                pass

        # B) Busca chamadas de ferramentas nas mensagens internas da mensagem final
        print("Searching for tool calls in final message...")
        for msg in getattr(resp.choices[0].message, "intermediate_messages", []) or []:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    calls.append((tc.function.name, tc.function.arguments))

        # Remove chamadas duplicadas para não repetir ferramentas na interface final
        seen = set()
        dedup_calls = []
        print("Removing duplicate tool calls...")
        for name, args in calls:
            key = (name, args)
            if key not in seen:
                seen.add(key)
                dedup_calls.append((name, args))

        # Formata os argumentos das ferramentas para ficarem legíveis (JSON -> Texto)
        print("Formatting tool arguments for readability...")
        tool_lines = []
        for name, args in dedup_calls:
            arg_text = str(args)
            try:
                import json as _json
                parsed = _json.loads(args) if isinstance(args, str) else args
                if isinstance(parsed, dict):
                    kv = ", ".join(f"{k}={repr(v)}" for k, v in parsed.items())
                    arg_text = kv
            except Exception:
                pass # Se não for JSON, mantém o texto bruto
            tool_lines.append(f"- {name}({arg_text})")

        print("Adding tool usage section to the output...")
        # Adiciona uma seção visual no final da resposta indicando quais ferramentas foram usadas
        if tool_lines:
            tools_html = (
                "<h2 style='font-size:1.5em; color:#2563eb;'>📎 Tools used</h2>"
            )
            tools_html += (
                "<ul>" + "".join(f"<li>{line}</li>" for line in tool_lines) + "</ul>"
            )
            content += "\n\n" + tools_html

        print("✅ Output:\n", content)
        return content, messages

    except Exception as e:
        print("❌ Error:", e)
        return f"[Model Error: {str(e)}]", messages

In [6]:
content, messages = research_agent(prompt="o profissional de marketing digital na era da IA")

start - 🔍 Research Agent
Calling the model...
Searching for tool calls in intermediate responses...
Searching for tool calls in final message...
Removing duplicate tool calls...
Formatting tool arguments for readability...
Adding tool usage section to the output...
✅ Output:
 - Title: IA no marketing digital: O início de uma nova era - Lisbon Digital School
- URL/Link: https://lisbondigitalschool.com/blog/ia-no-marketing-digital/
- Abstract: Os profissionais de marketing modernos devem considerar integrar a IA e Machine Learning (ML) nas suas estratégias digitais, recorrendo às ferramentas que essas tecnologias oferecem para otimizar campanhas e melhorar resultados.

- Title: Como os profissionais de marketing estão lidando com a IA?
- URL/Link: https://www.meioemensagem.com.br/marketing/profissionais-de-marketing-ia
- Abstract: A maioria (99%) dos respondentes disseram que a IA irá revolucionar o marketing. Essa é uma ideia que se sobressai entre os brasileiros, indicando uma forte te

In [ ]:
print(content)

- Title: IA no marketing digital: O início de uma nova era - Lisbon Digital School
- URL/Link: https://lisbondigitalschool.com/blog/ia-no-marketing-digital/
- Abstract: Os profissionais de marketing modernos devem considerar integrar a IA e Machine Learning (ML) nas suas estratégias digitais, recorrendo às ferramentas que essas tecnologias oferecem para otimizar campanhas e melhorar resultados.

- Title: Como os profissionais de marketing estão lidando com a IA?
- URL/Link: https://www.meioemensagem.com.br/marketing/profissionais-de-marketing-ia
- Abstract: A maioria (99%) dos respondentes disseram que a IA irá revolucionar o marketing. Essa é uma ideia que se sobressai entre os brasileiros, indicando uma forte tendência de adoção da IA no marketing digital.

- Title: O futuro do marketing digital: sua agência está pronta para a IA?
- URL/Link: https://projectcor.com/blog-portugues/o-futuro-do-marketing-digital-sua-agencia-esta-pronta-para-a-ia/
- Abstract: A IA automatiza tarefas repe

In [12]:
from typing import List, Dict, Optional
import os
import re
import time
import tempfile
import xml.etree.ElementTree as ET
from io import BytesIO
import wikipedia

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
# ----- Ferramenta de Pesquisa: arXiv -----
def _build_session(
    user_agent: str = "LF-ADP-Agent/1.0 (mailto:your.email@example.com)",
) -> requests.Session:
    """
    Cria e configura uma sessão do requests com uma política de retentativas (retry)
    robusta para lidar com falhas temporárias de rede ou limites de taxa (rate limits).
    """
    s = requests.Session()
    # Define cabeçalhos padrão para simular um navegador/agente real e evitar bloqueios
    s.headers.update(
        {
            "User-Agent": user_agent,
            "Accept": "*/*",
            "Accept-Encoding": "gzip, deflate",
            "Connection": "keep-alive",
        }
    )
    # Configura a estratégia de repetição: tenta até 5 vezes em caso de erros 429, 50x, etc.
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=0.6, # Tempo de espera exponencial entre as tentativas
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_redirect=False,
        raise_on_status=False,
    )
    # Monta o adaptador HTTP na sessão para conexões HTTP e HTTPS
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

# Inicializa uma sessão global para ser reutilizada pelas funções
session = _build_session()


# ----- Funções Utilitárias para Tratamento de PDFs e Textos -----
def ensure_pdf_url(abs_or_pdf_url: str) -> str:
    """
    Garante que a URL do arXiv aponte diretamente para o arquivo PDF 
    em vez da página de resumo (abstract).
    """
    url = abs_or_pdf_url.strip().replace("http://", "https://")
    if "/pdf/" in url and url.endswith(".pdf"):
        return url
    # Substitui a rota '/abs/' (abstract) por '/pdf/'
    url = url.replace("/abs/", "/pdf/")
    if not url.endswith(".pdf"):
        url += ".pdf"
    return url

def _safe_filename(name: str) -> str:
    """
    Remove caracteres especiais de uma string para torná-la um nome de arquivo seguro.
    """
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    if not name.lower().endswith(".pdf"):
        name += ".pdf"
    return name

def clean_text(s: str) -> str:
    """
    Limpa e normaliza o texto extraído de PDFs (remove quebras de palavras, 
    espaços extras e múltiplas quebras de linha).
    """
    s = re.sub(r"-\n", "", s)         # Une palavras separadas por hífen no final da linha
    s = re.sub(r"\r\n|\r", "\n", s)   # Normaliza os saltos de linha para o padrão Unix
    s = re.sub(r"[ \t]+", " ", s)     # Colapsa múltiplos espaços em um único espaço
    s = re.sub(r"\n{3,}", "\n\n", s)  # Evita mais de uma linha em branco seguida
    return s.strip()

def fetch_pdf_bytes(pdf_url: str, timeout: int = 90) -> bytes:
    """
    Faz o download do PDF em formato de bytes usando a sessão configurada.
    """
    r = session.get(pdf_url, timeout=timeout, allow_redirects=True)
    r.raise_for_status() # Levanta exceção se o status HTTP indicar erro
    return r.content

def pdf_bytes_to_text(pdf_bytes: bytes, max_pages: Optional[int] = None) -> str:
    """
    Extrai texto de um PDF em memória (bytes). 
    Tenta usar PyMuPDF (fitz) primeiro, e se falhar, tenta pdfminer.six.
    """
    # 1ª Tentativa: PyMuPDF (geralmente mais rápido e preciso)
    try:
        import fitz  # PyMuPDF
        out = []
        with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
            n = len(doc)
            limit = n if max_pages is None else min(max_pages, n)
            for i in range(limit):
                out.append(doc.load_page(i).get_text("text"))
        return "\n".join(out)
    except Exception:
        pass # Se falhar ou não estiver instalado, passa para o fallback

    # 2ª Tentativa: pdfminer.six (fallback)
    try:
        from pdfminer.high_level import extract_text_to_fp
        buf_in = BytesIO(pdf_bytes)
        buf_out = BytesIO()
        extract_text_to_fp(buf_in, buf_out)
        return buf_out.getvalue().decode("utf-8", errors="ignore")
    except Exception as e:
        raise RuntimeError(f"PDF text extraction failed: {e}")

def maybe_save_pdf(pdf_bytes: bytes, dest_dir: str, filename: str) -> str:
    """
    Salva os bytes do PDF fisicamente no disco em um diretório especificado.
    """
    os.makedirs(dest_dir, exist_ok=True)
    path = os.path.join(dest_dir, _safe_filename(filename))
    with open(path, "wb") as f:
        f.write(pdf_bytes)
    return path


def arxiv_search_tool(
    query: str,
    max_results: int = 3,
) -> List[Dict]:
    """
    Busca artigos no arXiv. Retorna uma lista de dicionários onde a chave `summary`
    é sobrescrita para conter o texto extraído diretamente do PDF do artigo.
    """
    # ===== FLAGS INTERNOS (Configurações da ferramenta) =====
    _INCLUDE_PDF = True       # Controla se o PDF deve ser processado
    _EXTRACT_TEXT = True      # Controla se o texto deve ser extraído do PDF
    _MAX_PAGES = 6            # Limita a extração às primeiras N páginas (economiza tempo/tokens)
    _TEXT_CHARS = 5000        # Limita o número de caracteres retornados do texto
    _SAVE_FULL_TEXT = False   # Se True, ignora _TEXT_CHARS e salva o texto completo
    _SLEEP_SECONDS = 1.0      # Pausa entre requisições para evitar rate limit do arXiv
    # ==========================

    # Constrói a URL da API do arXiv
    api_url = (
        "https://export.arxiv.org/api/query"
        f"?search_query=all:{requests.utils.quote(query)}&start=0&max_results={max_results}"
    )

    out: List[Dict] = []
    
    # Faz a requisição para a API
    try:
        resp = session.get(api_url, timeout=60)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        return [{"error": f"arXiv API request failed: {e}"}]

    # Analisa o XML retornado pelo arXiv
    try:
        root = ET.fromstring(resp.content)
        ns = {"atom": "http://www.w3.org/2005/Atom"} # Namespace do feed Atom

        for entry in root.findall("atom:entry", ns):
            # Extrai metadados básicos
            title = (entry.findtext("atom:title", default="", namespaces=ns) or "").strip()
            published = (entry.findtext("atom:published", default="", namespaces=ns) or "")[:10]
            url_abs = entry.findtext("atom:id", default="", namespaces=ns) or ""
            abstract_summary = (entry.findtext("atom:summary", default="", namespaces=ns) or "").strip()

            # Extrai autores
            authors = []
            for a in entry.findall("atom:author", ns):
                nm = a.findtext("atom:name", default="", namespaces=ns)
                if nm:
                    authors.append(nm)

            # Encontra o link direto para o PDF
            link_pdf = None
            for link in entry.findall("atom:link", ns):
                if link.attrib.get("title") == "pdf":
                    link_pdf = link.attrib.get("href")
                    break
            if not link_pdf and url_abs:
                link_pdf = ensure_pdf_url(url_abs)

            # Monta o dicionário de resultado para este artigo
            item = {
                "title": title,
                "authors": authors,
                "published": published,
                "url": url_abs,
                "summary": abstract_summary, # Começa com o abstract, pode ser sobrescrito abaixo
                "link_pdf": link_pdf,
            }

            pdf_bytes = None
            # Baixa o PDF se as flags permitirem e o link existir
            if (_INCLUDE_PDF or _EXTRACT_TEXT) and link_pdf:
                try:
                    pdf_bytes = fetch_pdf_bytes(link_pdf, timeout=90)
                    time.sleep(_SLEEP_SECONDS) # Pausa educada entre downloads
                except Exception as e:
                    item["pdf_error"] = f"PDF fetch failed: {e}"

            # Extrai o texto do PDF baixado
            if _EXTRACT_TEXT and pdf_bytes:
                try:
                    text = pdf_bytes_to_text(pdf_bytes, max_pages=_MAX_PAGES)
                    text = clean_text(text) if text else ""
                    if text:
                        # Substitui o `summary` (abstract) pelo texto real do PDF
                        if _SAVE_FULL_TEXT:
                            item["summary"] = text
                        else:
                            item["summary"] = text[:_TEXT_CHARS]
                except Exception as e:
                    item["text_error"] = f"Text extraction failed: {e}"

            out.append(item)
        return out
        
    except ET.ParseError as e:
        return [{"error": f"arXiv API XML parse failed: {e}"}]
    except Exception as e:
        return [{"error": f"Unexpected error: {e}"}]


In [30]:
test = arxiv_search_tool('quantum computing')

KeyboardInterrupt: 

In [21]:
for i in test:
    print(i)

{'title': 'Expected Performance of the ATLAS Experiment - Detector, Trigger and Physics', 'authors': [' The ATLAS Collaboration', 'G. Aad', 'E. Abat', 'B. Abbott', 'J. Abdallah', 'A. A. Abdelalim', 'A. Abdesselam', 'O. Abdinov', 'B. Abi', 'M. Abolins', 'H. Abramowicz', 'B. S. Acharya', 'D. L. Adams', 'T. N. Addy', 'C. Adorisio', 'P. Adragna', 'T. Adye', 'J. A. Aguilar-Saavedra', 'M. Aharrouche', 'S. P. Ahlen', 'F. Ahles', 'A. Ahmad', 'H. Ahmed', 'G. Aielli', 'T. Akdogan', 'T. P. A. Akesson', 'G. Akimoto', 'M. S. Alam', 'M. A. Alam', 'J. Albert', 'S. Albrand', 'M. Aleksa', 'I. N. Aleksandrov', 'F. Alessandria', 'C. Alexa', 'G. Alexander', 'G. Alexandre', 'T. Alexopoulos', 'M. Alhroob', 'G. Alimonti', 'J. Alison', 'M. Aliyev', 'P. P. Allport', 'S. E. Allwood-Spiers', 'A. Aloisio', 'R. Alon', 'A. Alonso', 'J. Alonso', 'M. G. Alviggi', 'K. Amako', 'P. Amaral', 'C. Amelung', 'V. V. Ammosov', 'A. Amorim', 'G. Amoros', 'N. Amram', 'C. Anastopoulos', 'C. F. Anders', 'K. J. Anderson', 'A. Andre

In [39]:

# ===== FLAGS INTERNOS (Configurações da ferramenta) =====
_INCLUDE_PDF = True       # Controla se o PDF deve ser processado
_EXTRACT_TEXT = True      # Controla se o texto deve ser extraído do PDF
_MAX_PAGES = 6            # Limita a extração às primeiras N páginas (economiza tempo/tokens)
_TEXT_CHARS = 5000        # Limita o número de caracteres retornados do texto
_SAVE_FULL_TEXT = False   # Se True, ignora _TEXT_CHARS e salva o texto completo
_SLEEP_SECONDS = 1.0      # Pausa entre requisições para evitar rate limit do arXiv
# ==========================

query = 'quantum computing'
max_results = 3

# Use uma sessão para melhor performance
session = requests.Session()

# Constrói a URL corretamente
encoded_query = requests.utils.quote(query)
api_url = f"https://export.arxiv.org/api/query?search_query=all:{encoded_query}&start=0&max_results={max_results}"

out: List[Dict] = []

try:
    # Faz a requisição
    resp = session.get(api_url, timeout=60)
    resp.raise_for_status()
    
    # O SEGREDO: Use resp.text para pegar a string XML já decodificada
    xml_data = resp.text 
    
    # Agora você pode processar o xml_data com o ElementTree ou enviar para o Agente
    print("XML capturado com sucesso!")

except requests.exceptions.RequestException as e:
    print(f"Erro na requisição: {e}")
    out = [{"error": f"arXiv API request failed: {e}"}]
    [{"error": f"arXiv API request failed: {e}"}]

XML capturado com sucesso!


In [57]:

import time

def get_paper_metrics(arxiv_id: str):
    """
    Busca métricas de impacto para um ID do arXiv.
    O formato do ID deve ser apenas os números (ex: 2211.02350)
    """
    # Remove versões do ID se existirem (ex: 2211.02350v1 -> 2211.02350)
    clean_id = arxiv_id.split('v')[0]
    
    url = f"https://api.semanticscholar.org/graph/v1/paper/arXiv:{clean_id}"
    params = {'fields': 'citationCount,influentialCitationCount,year,title'}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return {
                "citations": data.get("citationCount", 0),
                "influential": data.get("influentialCitationCount", 0)
            }
    except Exception as e:
        print(f"Erro ao buscar Semantic Scholar: {e}")
    
    return  {"citations": 0, "influential": 0}

# Exemplo de uso no seu loop de artigos:
# metrics = get_paper_metrics("2211.02350")
# print(f"Citações: {metrics['citations']} | Influentes: {metrics['influential']}")
metrics = get_paper_metrics('2211.02350v1')

In [63]:
url = f"https://api.semanticscholar.org/graph/v1/paper/arXiv:2211.02350"
response = requests.get(url, timeout=10)
data = response.json()

In [64]:
data

{'message': 'Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form',
 'code': '429'}